# Rotational Cooling Terminal-Event Scans

This notebook uses the same `P2`, `J12`, and `J23` transition set from `rotational_cooling_parameter_scans.ipynb`. The laser remains `Z` polarized, while the two microwave drives use square-wave polarization switching with one shared microwave polarization clock.

The scan metric is the time required to reach summed `X, J=0` population `0.95` from a normalized `T = 6.3 K` thermal initial state. Each trajectory can run up to `1000 us`; scan points that do not reach the threshold are shown as `1000 us`.


In [ ]:
from __future__ import annotations

from itertools import combinations
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np

from centrex_tlf import couplings, hamiltonian, lindblad, states, transitions
from centrex_tlf.lindblad import (
    LindbladParameters,
    grid_scan,
    prepare_lindblad_problem,
    resonant_polarization_modulation,
    square_wave,
)
from centrex_tlf.lindblad.events import population
from centrex_tlf.lindblad.solve import solve_lindblad
from centrex_tlf.utils.population import generate_thermal_population_states

plt.rcParams.update({"font.size": 14})

Gamma = 2.0 * np.pi * 1.56e6
temperature = 6.3
t_final = 1000e-6
threshold_population = 0.95
t_eval = np.linspace(0.0, t_final, 1001)

scan_threads = None
scan_points = 10
scan_min = 0.1
scan_max = 3.0
scan_axis = np.linspace(scan_min, scan_max, scan_points)

default_values = {
    "Omega_mw_12": 1.0,
    "Omega_mw_23": 1.0,
    "Omega_laser": 1.0,
    "omega_mw_pol": 1.0,
}

solver_options = {
    "solver": "dopri5",
    "execution_mode": "expanded_sparse",
    "abstol": 1e-6,
    "reltol": 1e-3,
    "dt": 1e-10,
    "maxiters": 5_000_000,
}


## System And Parameters

The microwave selectors use the same clock with a fixed phase offset:

- `J12`: `square_wave(t, omega_mw_pol, 0)`
- `J23`: `square_wave(t, omega_mw_pol, pi / 2)`
- each second polarization gets the complement `1 - square_wave(...)`

In [ ]:
P2 = transitions.OpticalTransition(transitions.OpticalTransitionType.P, 2, 3 / 2, 1)
J12 = transitions.MicrowaveTransition(1, 2)
J23 = transitions.MicrowaveTransition(2, 3)
transition_set = [P2, J12, J23]

mw12_a = (couplings.polarization_X - couplings.polarization_Z).normalize()
mw12_b = couplings.polarization_Y
mw23_a = (couplings.polarization_X + couplings.polarization_Z).normalize()
mw23_b = couplings.polarization_Y


def build_rotational_cooling_system(*, laser_switching: bool = False):
    laser_polarizations = (
        [couplings.polarization_σp, couplings.polarization_σm]
        if laser_switching
        else [couplings.polarization_Z]
    )
    selectors = couplings.generate_transition_selectors(
        transition_set,
        [
            laser_polarizations,
            [mw12_a, mw12_b],
            [mw23_a, mw23_b],
        ],
    )
    system = lindblad.generate_OBE_system_transitions(
        transition_set,
        selectors,
        method="matrix",
    )
    return system, selectors


def make_parameters(selectors, *, laser_switching: bool = False):
    params = LindbladParameters()
    time = params.time()

    omega_laser = params.real("Omega_laser", default_values["Omega_laser"] * Gamma)
    omega_mw_12 = params.real("Omega_mw_12", default_values["Omega_mw_12"] * Gamma)
    omega_mw_23 = params.real("Omega_mw_23", default_values["Omega_mw_23"] * Gamma)
    omega_mw_pol = params.real("omega_mw_pol", default_values["omega_mw_pol"] * Gamma)

    params.drive(selectors[0], rabi=omega_laser, detuning=0.0)
    params.drive(selectors[1], rabi=omega_mw_12, detuning=0.0)
    params.drive(selectors[2], rabi=omega_mw_23, detuning=0.0)

    if laser_switching:
        mod_depth = params.real("mod_depth", np.pi / 2)
        omega_laser_pol = omega_mw_pol / 2
        params.bind(
            selectors[0].polarization_symbols[0],
            resonant_polarization_modulation(time, -mod_depth, omega_laser_pol),
            finalize=False,
        )
        params.bind(
            selectors[0].polarization_symbols[1],
            resonant_polarization_modulation(time, mod_depth, omega_laser_pol),
            finalize=False,
        )
    else:
        params.bind(selectors[0].polarization_symbols[0], 1.0, finalize=False)

    mw12_gate = square_wave(time, omega_mw_pol, 0.0)
    mw23_gate = square_wave(time, omega_mw_pol, np.pi / 2)
    params.bind(selectors[1].polarization_symbols[0], mw12_gate, finalize=False)
    params.bind(selectors[1].polarization_symbols[1], 1.0 - mw12_gate, finalize=False)
    params.bind(selectors[2].polarization_symbols[0], mw23_gate, finalize=False)
    params.bind(selectors[2].polarization_symbols[1], 1.0 - mw23_gate, finalize=False)
    params._finalize()

    knobs = {
        "Omega_mw_12": omega_mw_12,
        "Omega_mw_23": omega_mw_23,
        "Omega_laser": omega_laser,
        "omega_mw_pol": omega_mw_pol,
    }
    return params, knobs


def prepare_system(*, laser_switching: bool = False):
    system, selectors = build_rotational_cooling_system(laser_switching=laser_switching)
    params, knobs = make_parameters(selectors, laser_switching=laser_switching)
    prepared = prepare_lindblad_problem(
        system,
        params,
        backend="rust",
        hamiltonian_representation="decomposed",
    )
    return system, selectors, params, knobs, prepared

In [ ]:
def normalized_thermal_rho(system):
    rho0 = generate_thermal_population_states(temperature, system.QN)
    rho0 = rho0 / np.trace(rho0)
    return rho0


def dominant_basis_state(state):
    return getattr(state, "largest", state)


def is_x_state(state):
    return dominant_basis_state(state).electronic_state == states.ElectronicState.X


def is_x_j_state(state, j_value):
    basis = dominant_basis_state(state)
    return basis.electronic_state == states.ElectronicState.X and basis.J == j_value


def state_label(state, quantum_numbers=("J", "F1", "F", "mF")):
    basis = dominant_basis_state(state)
    if hasattr(basis, "state_string_custom"):
        return basis.state_string_custom(list(quantum_numbers))
    if hasattr(basis, "state_string"):
        return basis.state_string()
    return str(basis)


def indices_for_x_j(system, j_value):
    return [idx for idx, state in enumerate(system.QN) if is_x_j_state(state, j_value)]


def j_groups(system):
    groups = {}
    for idx, state in enumerate(system.QN):
        if is_x_state(state):
            groups.setdefault(dominant_basis_state(state).J, []).append(idx)
    return dict(sorted(groups.items()))


def populations_by_j(populations, system):
    groups = j_groups(system)
    return {j: np.sum(populations[:, indices], axis=1) for j, indices in groups.items()}


def final_populations_by_j(populations, system):
    return {j: values[-1] for j, values in populations_by_j(populations, system).items()}


def terminal_time_grid_us(result):
    grid_shape = tuple(result.metadata["grid_shape"])
    return np.asarray(result.t, dtype=float).reshape(grid_shape) * 1e6


def event_triggered_grid(result):
    grid_shape = tuple(result.metadata["grid_shape"])
    return np.asarray(result.metadata["event_triggered"], dtype=bool).reshape(grid_shape)


def selected_population_grid(result, target_indices):
    grid_shape = tuple(result.metadata["grid_shape"])
    values = np.real_if_close(result.values).real
    return values.reshape((*grid_shape, len(target_indices))).sum(axis=-1)


def heatmap(ax, grid, x_axis, y_axis, xlabel, ylabel, title):
    image = ax.imshow(
        grid.T,
        origin="lower",
        aspect="auto",
        extent=[x_axis[0], x_axis[-1], y_axis[0], y_axis[-1]],
        cmap="viridis_r",
        vmin=0.0,
        vmax=t_final * 1e6,
    )
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    return image


scan_definitions = {
    "Omega_mw_12": {"label": r"$\Omega_{12}/\Gamma$", "axis": scan_axis},
    "Omega_mw_23": {"label": r"$\Omega_{23}/\Gamma$", "axis": scan_axis},
    "Omega_laser": {"label": r"$\Omega_\mathrm{laser}/\Gamma$", "axis": scan_axis},
    "omega_mw_pol": {"label": r"$\omega_\mathrm{mw\ pol}/\Gamma$", "axis": scan_axis},
}
scan_pairs = list(combinations(scan_definitions, 2))


def run_scan_slice(prepared, rho0, knobs, target_indices, stop_event, pair, *, axis_override=None):
    axis = scan_axis if axis_override is None else np.asarray(axis_override, dtype=float)
    first, second = pair
    scan = {
        knobs[first]: axis * Gamma,
        knobs[second]: axis * Gamma,
    }
    start = perf_counter()
    result = grid_scan(
        prepared,
        rho0,
        (0.0, t_final),
        scan=scan,
        output="selected",
        output_indices=[(idx, idx) for idx in target_indices],
        output_when="final",
        stop_event=stop_event,
        dense_output=False,
        parallel=True,
        collect_stats=True,
        threads=scan_threads,
        **solver_options,
    )
    time_grid_us = terminal_time_grid_us(result)
    triggered_grid = event_triggered_grid(result)
    population_grid = selected_population_grid(result, target_indices)
    return {
        "pair": pair,
        "axis": axis,
        "grid": time_grid_us,
        "event_triggered": triggered_grid,
        "terminal_population": population_grid,
        "elapsed_s": perf_counter() - start,
        "result": result,
    }


def best_point(slice_result):
    grid = slice_result["grid"]
    axis = slice_result["axis"]
    pair = slice_result["pair"]
    best = np.unravel_index(np.nanargmin(grid), grid.shape)
    return {
        "slice": f"{pair[0]} vs {pair[1]}",
        pair[0]: axis[best[0]],
        pair[1]: axis[best[1]],
        "time_us": grid[best],
        "reached_threshold": bool(slice_result["event_triggered"][best]),
        "terminal_X,J=0": slice_result["terminal_population"][best],
        "elapsed_s": slice_result["elapsed_s"],
    }


In [ ]:
system, selectors, params, knobs, prepared = prepare_system(laser_switching=False)
rho0 = normalized_thermal_rho(system)

target_indices = indices_for_x_j(system, 0)
if not target_indices:
    raise RuntimeError("No X, J=0 target states were found.")
j0_stop_event = population(
    target_indices,
    threshold=threshold_population,
    name=f"X,J=0 >= {threshold_population:.2f}",
)

print(f"Transitions: {[transition.name for transition in transition_set]}")
print(f"Laser polarization symbols: {selectors[0].polarization_symbols}")
print(
    f"Microwave polarization symbols: {[selectors[1].polarization_symbols, selectors[2].polarization_symbols]}"
)
print(f"System size: {len(system.QN)} states")
print(f"Trace(rho0): {np.trace(rho0):.12f}")
print(f"Target X, J=0 indices: {target_indices}")
print(f"Example target labels: {[state_label(system.QN[idx]) for idx in target_indices[:5]]}")
print(f"Threshold event: summed X, J=0 population >= {threshold_population:.2f}")
print(f"Maximum interaction time: {t_final * 1e6:.0f} us")
print(f"Free scan knobs: {list(knobs)}")


## Baseline Terminal Evolution

The baseline uses the top-level defaults: all Rabi rates and the microwave polarization switching frequency are `1 Gamma`. The solver stops early if the summed `X, J=0` population reaches the threshold.


In [ ]:
baseline = solve_lindblad(
    prepared,
    rho0,
    (0.0, t_final),
    saveat=t_eval,
    output="populations",
    output_when="saveat",
    stop_event=j0_stop_event,
    dense_output=True,
    collect_stats=True,
    **solver_options,
)
baseline_populations = baseline.values
baseline_by_j = populations_by_j(baseline_populations, system)
baseline_j0 = baseline_populations[:, target_indices].sum(axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
for j_value, values in baseline_by_j.items():
    ax.plot(1e6 * baseline.t, values, label=f"X, J={j_value}")
ax.axhline(threshold_population, color="black", ls="--", lw=1.0, label="J=0 threshold")
ax.set_xlabel("time (us)")
ax.set_ylabel("population")
ax.set_title("Rotational population evolution with terminal event")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

final_summary = final_populations_by_j(baseline_populations, system)
event_triggered = bool(baseline.solver_stats and baseline.solver_stats.get("event_triggered"))
print("Terminal rotational population summary")
for j_value, population_value in final_summary.items():
    print(f"X, J={j_value}: {population_value:.6f}")
print(f"Terminal X, J=0 target population: {baseline_j0[-1]:.6f}")
print(f"Terminal time: {baseline.t[-1] * 1e6:.3f} us")
print(f"Event triggered: {event_triggered}")
if event_triggered:
    print(f"Event time from solver stats: {baseline.solver_stats['event_time'] * 1e6:.3f} us")
print(baseline.solver_stats)


## Pairwise 2D Time-To-Threshold Scans

Each slice scans two parameters from `0.1 Gamma` to `3 Gamma` with `10 x 10` points, holding the other two parameters at their defaults. The plotted value is the terminal time in microseconds. Points that do not reach the threshold before `1000 us` appear at `1000 us`.


In [ ]:
scan_results = {}
best_points = []
for pair in scan_pairs:
    print(f"Running slice: {pair[0]} vs {pair[1]}")
    slice_result = run_scan_slice(prepared, rho0, knobs, target_indices, j0_stop_event, pair)
    scan_results[pair] = slice_result
    best = best_point(slice_result)
    best_points.append(best)
    reached = int(np.count_nonzero(slice_result["event_triggered"]))
    total = slice_result["event_triggered"].size
    print(best)
    print(f"Reached threshold: {reached}/{total}")


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 9), constrained_layout=True)
for ax, pair in zip(axes.ravel(), scan_pairs, strict=True):
    result = scan_results[pair]
    axis = result["axis"]
    image = heatmap(
        ax,
        result["grid"],
        axis,
        axis,
        scan_definitions[pair[0]]["label"],
        scan_definitions[pair[1]]["label"],
        f"{pair[0]} vs {pair[1]}",
    )
    fig.colorbar(image, ax=ax, label="time to 0.95 J=0 population (us)")
plt.show()

best_points


## Notes

The terminal event is evaluated on the summed diagonal populations for all `X, J=0` basis states. `grid_scan` returns one terminal time per scan point when `stop_event` is used with `output_when="final"`, so `result.t` is the scan metric here. The selected final populations are retained in each `scan_results[pair]["terminal_population"]` entry as a diagnostic.
